# 🔵 Tropical Flower Classification — CNN Architectures Deep Dive (Not Optimized Solution) 

**Dataset**: Tropical Flower Dataset  
**Models**: MobileNetV3-Large · ResNeXt-50-32×4d · EfficientNet-B3  

## 📋 Table of Contents
1. [⚙️ Global Configuration (START HERE)](#config)
2. [📦 Setup & Imports](#setup)
3. [📊 Exploratory Data Analysis](#eda)
4. [🔧 Data Preprocessing & Augmentation](#preprocessing)
5. [🏋️ Training Infrastructure](#training)
6. [🔵 CNN Models: MobileNetV3 · ResNeXt-50 · EfficientNet-B3](#cnn)
7. [📈 Evaluation & Comparison](#evaluation)
8. [🛡️ Robustness Analysis](#robustness)
9. [🔍 Error Analysis](#error)



## ⚙️ Global Configuration ! <a id='config'></a>

**All hyperparameters and toggles are centralized here.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║           GLOBAL CONFIGURATION — EDIT THIS CELL ONLY            ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Dataset ────────────────────────────────────────────────────────
BASE_DIR   = '/kaggle/input/tropical-flowers'
OUTPUT_DIR = '/kaggle/working'

# ── Pretrained weights ─────────────────────────────────────────────
# Kaggle blocks internet at runtime → pretrained=True will fail.
# FIX A (best): In your notebook, go to Data → Add Data → search
#   "timm pretrained" or "pytorch image models" and add that dataset.
#   Then set USE_PRETRAINED = True.
# FIX B (quick): Set USE_PRETRAINED = False (trains from scratch).
USE_PRETRAINED = False   # ← Change to True if you added weights dataset

# ── Image settings ─────────────────────────────────────────────────
IMG_SIZE    = 224    # Input resolution. Options: 128, 224, 256, 384
                     # Higher = better accuracy but slower training

# ── Training ───────────────────────────────────────────────────────
BATCH_SIZE      = 32     # Reduce to 16 if you get CUDA out-of-memory
NUM_EPOCHS      = 20     # Epochs per model
ABLATION_EPOCHS = 1     # Epochs for ablation experiments
SEED            = 42

# ── Optimizer ──────────────────────────────────────────────────────
CNN_LR          = 3e-4   # Learning rate for CNN models
TRANSFORMER_LR  = 1e-4   # Transformers need smaller LR
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0    # Max gradient norm for gradient clipping

# ── Loss function ──────────────────────────────────────────────────
LABEL_SMOOTHING     = 0.1    # 0 = standard CE, 0.1 = slight smoothing
USE_CLASS_WEIGHTS   = True   # Weight loss by inverse class frequency

# ── Regularization ─────────────────────────────────────────────────
DROPOUT_RATE    = 0.3    # Dropout before classifier head
MIXUP_ALPHA     = 0.2    # MixUp augmentation strength (0 = disabled)
EARLY_STOP_PAT  = 6      # Patience for early stopping (epochs)

# ── Augmentation ───────────────────────────────────────────────────
RANDAUG_OPS     = 2      # RandAugment: number of operations
RANDAUG_MAG     = 9      # RandAugment: magnitude (1-30)
RANDOM_ERASE_P  = 0.1    # Probability of Random Erasing (Cutout)

# ── Model selection ────────────────────────────────────────────────
# CNN timm model IDs (these are exact strings used in timm.create_model)
CNN_MODELS = {
    'MobileNetV3-Large': 'mobilenetv3_large_100',
    'ResNeXt-50-32x4d':  'resnext50_32x4d',
    'EfficientNet-B3':   'efficientnet_b3',
}

# Transformer timm model IDs
TRANSFORMER_MODELS = {
    'ViT-B/16':   'vit_base_patch16_224',
    'Swin-Tiny':  'swin_tiny_patch4_window7_224',
    'DeiT-Small': 'deit_small_patch16_224',
}

# ── GradCAM ────────────────────────────────────────────────────────
GRADCAM_SAMPLES_PER_CLASS = 2   # Images to visualize per class

# ── Robustness ─────────────────────────────────────────────────────
NOISE_LEVELS = [0.05, 0.15]   # Gaussian noise σ values to test
BLUR_KERNELS = [3, 9]         # Blur kernel sizes to test

# ── Display ────────────────────────────────────────────────────────
COLORS      = ['#E74C3C', '#F39C12', '#8E44AD', '#27AE60']
FIGDPI      = 120

# ── ImageNet normalization (DO NOT CHANGE) ──────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("✅ Configuration loaded successfully!")
print(f"   IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={NUM_EPOCHS}")
print(f"   USE_PRETRAINED={USE_PRETRAINED}")
print(f"   CNN LR={CNN_LR}, Transformer LR={TRANSFORMER_LR}")
print(f"   Label smoothing={LABEL_SMOOTHING}, Class weighting={USE_CLASS_WEIGHTS}")
print("\nCNN Models:")
for name, tid in CNN_MODELS.items():
    print(f"   {name:<22} → timm id: '{tid}'")
print("\nTransformer Models:")
for name, tid in TRANSFORMER_MODELS.items():
    print(f"   {name:<22} → timm id: '{tid}'")

## 📦 Setup & Imports <a id='setup'></a>

In [ ]:
# ── Install dependencies ────────────────────────────────────────────
# timm: PyTorch Image Models — ViT, Swin, DeiT, EfficientNet, etc.
# grad-cam: GradCAM / GradCAM++ for visualizing what the model sees
!pip install timm grad-cam -q

In [ ]:
import os, random, time, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F

# TorchVision
from torchvision import datasets, transforms, models

# timm
import timm

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# GradCAM
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Reproducibility ────────────────────────────────────────────────
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\n📦 PyTorch {torch.__version__} | timm {timm.__version__}")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
TRAIN_PATH = Path(BASE_DIR) / 'train'
TEST_PATH  = Path(BASE_DIR) / 'test'

CLASS_NAMES = sorted([d.name for d in TRAIN_PATH.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

print("📂 Classes:")
for i, c in enumerate(CLASS_NAMES):
    print(f"   [{i}] {c}")
print(f"\nTotal: {NUM_CLASSES} classes")

## 📊 Exploratory Data Analysis <a id='eda'></a>

> **Why EDA first?**  
> We need to understand **class imbalance** (→ weighted loss), typical **image dimensions** (→ resize strategy),
> and **visual appearance** of each disease before choosing any model.

In [ ]:
# ── Count images per split ─────────────────────────────────────────
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def count_images(base_path):
    return {
        d.name: len([f for f in d.iterdir() if f.suffix.lower() in EXTS])
        for d in sorted(base_path.iterdir()) if d.is_dir()
    }

train_counts = count_images(TRAIN_PATH)
test_counts  = count_images(TEST_PATH)

stats_df = pd.DataFrame({
    'Class': list(train_counts.keys()),
    'Train': list(train_counts.values()),
    'Test':  [test_counts.get(k, 0) for k in train_counts]
})
stats_df['Total']   = stats_df['Train'] + stats_df['Test']
stats_df['Train %'] = (stats_df['Train'] / stats_df['Train'].sum() * 100).round(1)

print(stats_df.to_string(index=False))
print(f"\nTrain total: {stats_df['Train'].sum()} | Test total: {stats_df['Test'].sum()}")
imbalance_ratio = stats_df['Train'].min() / stats_df['Train'].max()
print(f"Imbalance ratio (min/max): {imbalance_ratio:.2f}  "
      f"{'⚠️  Consider weighted loss' if imbalance_ratio < 0.7 else '✅ Relatively balanced'}")

In [ ]:
# ── Class distribution plots ───────────────────────────────────────
short = [c.replace(' Disease','').replace(' Leaf','') for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Tropical Flower Dataset — Class Distribution', fontsize=15, fontweight='bold')

# Grouped bar
x, w = np.arange(NUM_CLASSES), 0.35
axes[0].bar(x - w/2, stats_df['Train'], w, color=COLORS, alpha=0.85, label='Train')
axes[0].bar(x + w/2, stats_df['Test'],  w, color=COLORS, alpha=0.45, hatch='//', label='Test')
axes[0].set_xticks(x); axes[0].set_xticklabels(short, rotation=20, ha='right')
axes[0].set_title('Train vs Test'); axes[0].legend(); axes[0].set_ylabel('Count')
for b in axes[0].patches[:NUM_CLASSES]:
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1,
                 str(int(b.get_height())), ha='center', fontsize=8)

# Pie
axes[1].pie(stats_df['Train'], labels=short, colors=COLORS,
            autopct='%1.1f%%', startangle=90, pctdistance=0.8)
axes[1].set_title('Training Distribution')

# Imbalance bar
ratio = stats_df['Train'] / stats_df['Train'].max()
bars = axes[2].barh(short, ratio, color=COLORS, alpha=0.85)
axes[2].axvline(1.0, color='k', ls='--', alpha=0.4, label='Majority class')
axes[2].set_xlabel('Relative proportion'); axes[2].set_title('Imbalance Ratio')
axes[2].set_xlim(0, 1.2); axes[2].legend()
for bar, v in zip(bars, ratio):
    axes[2].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                 f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample images per class ────────────────────────────────────────
def sample_imgs(base, cls, n=5):
    files = [f for f in (base/cls).iterdir() if f.suffix.lower() in EXTS]
    return random.sample(files, min(n, len(files)))

N = 5
fig, axes = plt.subplots(NUM_CLASSES, N, figsize=(N*3, NUM_CLASSES*3))
fig.suptitle('Sample Images per Class (Training Set)', fontsize=14, fontweight='bold')

for r, cls in enumerate(CLASS_NAMES):
    for c, p in enumerate(sample_imgs(TRAIN_PATH, cls, N)):
        axes[r,c].imshow(Image.open(p).convert('RGB'))
        axes[r,c].axis('off')
        if c == 0:
            axes[r,c].set_ylabel(cls, fontsize=8, labelpad=48, rotation=0,
                                  ha='right', va='center', color=COLORS[r], fontweight='bold')
        if r == 0:
            axes[r,c].set_title(f'#{c+1}', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Image dimension & pixel statistics ────────────────────────────
print("⏳ Analyzing image properties (sampling 30 images/class)...")
records = []
for cls in CLASS_NAMES:
    for p in sample_imgs(TRAIN_PATH, cls, 30):
        img = Image.open(p).convert('RGB')
        arr = np.array(img)
        records.append({'class': cls, 'width': img.size[0], 'height': img.size[1],
                         'aspect': img.size[0]/img.size[1],
                         'mean_r': arr[:,:,0].mean(), 'mean_g': arr[:,:,1].mean(),
                         'mean_b': arr[:,:,2].mean(), 'std':    arr.std()})
props = pd.DataFrame(records)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Image Property Analysis', fontsize=13, fontweight='bold')

for i, cls in enumerate(CLASS_NAMES):
    sub = props[props['class']==cls]
    axes[0,0].hist(sub['width'],  alpha=0.6, color=COLORS[i], label=short[i], bins=15)
    axes[0,1].hist(sub['height'], alpha=0.6, color=COLORS[i], label=short[i], bins=15)
axes[0,0].set_title('Width Distribution');  axes[0,0].legend(fontsize=8)
axes[0,1].set_title('Height Distribution'); axes[0,1].legend(fontsize=8)

axes[0,2].boxplot([props[props['class']==c]['aspect'].values for c in CLASS_NAMES],
                   labels=short)
axes[0,2].set_title('Aspect Ratio'); axes[0,2].tick_params(axis='x', rotation=20)

props.groupby('class')[['mean_r','mean_g','mean_b']].mean().plot(
    kind='bar', ax=axes[1,0], color=['red','green','blue'], alpha=0.7)
axes[1,0].set_title('Mean RGB per Class'); axes[1,0].tick_params(axis='x', rotation=30)

for i, ch in enumerate(['mean_r','mean_g','mean_b']):
    axes[1,1].hist(props[ch], alpha=0.6, color=['red','green','blue'][i],
                   label=['R','G','B'][i], bins=20)
axes[1,1].set_title('Overall Channel Distribution'); axes[1,1].legend()

axes[1,2].boxplot([props[props['class']==c]['std'].values for c in CLASS_NAMES], labels=short)
axes[1,2].set_title('Pixel Std (Texture Complexity)'); axes[1,2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/image_properties.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()
print(f"Avg size: {props['width'].mean():.0f}×{props['height'].mean():.0f}px")

## 🔧 Data Preprocessing & Augmentation <a id='preprocessing'></a>

**Why ImageNet normalization?**  
Pretrained models were trained on ImageNet (μ=`[0.485,0.456,0.406]`, σ=`[0.229,0.224,0.225]`).  
Applying the same normalization ensures the input distribution matches what the backbone expects.

**Augmentation pipeline rationale:**

| Transform | Purpose |
|-----------|--------|
| `RandomCrop` | Scale invariance |
| `RandomHorizontalFlip` | Reflection symmetry |
| `ColorJitter` | Illumination invariance |
| `RandAugment` | Automatically learned policy for extra diversity |
| `RandomErasing` | Forces model to use full image, not single spot |

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandAugment(num_ops=RANDAUG_OPS, magnitude=RANDAUG_MAG),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=RANDOM_ERASE_P),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Datasets & DataLoaders ─────────────────────────────────────────
train_dataset = datasets.ImageFolder(str(TRAIN_PATH), transform=train_transform)
test_dataset  = datasets.ImageFolder(str(TEST_PATH),  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

# ── Class weights (for handling imbalance) ─────────────────────────
counts_arr      = np.array([train_counts[c] for c in CLASS_NAMES], dtype=float)
class_weights   = (1.0 / counts_arr) / (1.0 / counts_arr).sum() * NUM_CLASSES
CW_TENSOR       = torch.FloatTensor(class_weights).to(DEVICE)

print(f"✅ Train: {len(train_dataset)} images | {len(train_loader)} batches")
print(f"✅ Test : {len(test_dataset)} images | {len(test_loader)} batches")
print(f"⚖️  Class weights: {class_weights.round(3)}")

In [ ]:
# ── Visualize augmented batch ──────────────────────────────────────
def denorm(t):
    m = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    s = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(t * s + m, 0, 1)

imgs_b, lbls_b = next(iter(train_loader))
fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Augmented Training Samples — same image can look very different each epoch!',
             fontsize=11, fontweight='bold')
for idx in range(min(32, BATCH_SIZE)):
    r, c = idx//8, idx%8
    axes[r,c].imshow(denorm(imgs_b[idx]).permute(1,2,0).numpy())
    axes[r,c].set_title(CLASS_NAMES[lbls_b[idx]].split()[0], fontsize=7)
    axes[r,c].axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmented_samples.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

## 🏋️ Training Infrastructure <a id='training'></a>

In [ ]:
# ── Training loop utilities ────────────────────────────────────────

def build_criterion():
    """Build loss function using CONFIG values."""
    return nn.CrossEntropyLoss(
        weight=CW_TENSOR if USE_CLASS_WEIGHTS else None,
        label_smoothing=LABEL_SMOOTHING
    )

def train_epoch(model, loader, criterion, optimizer, scheduler=None):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        loss_sum += loss.item()
        correct  += model(images).argmax(1).eq(labels).sum().item()
        total    += labels.size(0)
    if scheduler: scheduler.step()
    return loss_sum / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Returns (loss, accuracy, preds, labels, probs)."""
    model.eval()
    if criterion is None: criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        out  = model(images)
        loss_sum += criterion(out, labels).item()
        probs     = F.softmax(out, dim=1)
        preds     = out.argmax(1)
        correct  += preds.eq(labels).sum().item()
        total    += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return (loss_sum/len(loader), 100.*correct/total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_model(model, name, lr=CNN_LR, epochs=NUM_EPOCHS):
    """
    Full training loop.
    Returns (best_model, history_dict, best_val_accuracy)
    """
    model = model.to(DEVICE)
    criterion = build_criterion()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_acc, patience_ctr = 0.0, 0

    print(f"\n{'='*65}")
    print(f" Training: {name}")
    print(f"{'='*65}")
    print(f"{'Ep':>4} | {'TrLoss':>8} | {'TrAcc':>7} | {'VaLoss':>8} | {'VaAcc':>7} | {'LR':>8}")
    print('-'*55)

    for ep in range(1, epochs+1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler)
        va_loss, va_acc, _, _, _ = evaluate(model, test_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)

        flag = ' ⭐' if va_acc > best_acc else ''
        print(f"{ep:>4} | {tr_loss:>8.4f} | {tr_acc:>6.2f}% | "
              f"{va_loss:>8.4f} | {va_acc:>6.2f}%{flag} | "
              f"{optimizer.param_groups[0]['lr']:.2e}  [{time.time()-t0:.1f}s]")

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), f'{OUTPUT_DIR}/{name}_best.pth')
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= EARLY_STOP_PAT:
                print(f"  ⏹️  Early stop at epoch {ep}")
                break

    model.load_state_dict(torch.load(f'{OUTPUT_DIR}/{name}_best.pth'))
    print(f"\n✅ Best val acc: {best_acc:.2f}%")
    return model, history, best_acc


def plot_history(history, name):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'{name} — Training History', fontsize=12, fontweight='bold')
    ep = range(1, len(history['val_acc'])+1)
    a1.plot(ep, history['train_loss'], 'b-o', ms=3, label='Train')
    a1.plot(ep, history['val_loss'],   'r-o', ms=3, label='Val')
    a1.set_title('Loss'); a1.legend(); a1.grid(alpha=.3)
    a2.plot(ep, history['train_acc'], 'b-o', ms=3, label='Train')
    a2.plot(ep, history['val_acc'],  'r-o', ms=3, label='Val')
    a2.set_title('Accuracy (%)'); a2.legend(); a2.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{name}_history.png', dpi=FIGDPI, bbox_inches='tight')
    plt.show()

print("✅ Training utilities defined.")

## 🔵 CNN Transfer Learning Models <a id='cnn'></a>

### Architecture Quick-Reference

| Model | Year | Params | Core Idea |
|-------|------|--------|-----------|
| **MobileNetV3-Large** | 2019 | 5.4M | Depthwise separable conv + SE + h-swish. Designed for mobile edge devices. |
| **ResNeXt-50-32×4d** | 2017 | 25M | ResNet + grouped convolutions. "Cardinality" (32 groups) as new dimension to scale. |
| **EfficientNet-B3** | 2019 | 12M | Compound scaling of depth/width/resolution. Best accuracy-per-FLOP in its era. |

> **Pretrained weight note**: `USE_PRETRAINED` is read from Config. If `False`, models train from scratch — expect ~10-15% lower accuracy but it will still run without internet.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CNN Model 1: MobileNetV3-Large                                  ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  Architecture:                                                    ║
# ║  - Inverted residual blocks (like MobileNetV2)                    ║
# ║  - Squeeze-and-Excitation (SE) in later layers                   ║
# ║  - Hard-Swish activation (approximates Swish, avoids exp())      ║
# ║  - NetAdapt search for optimal layer configs                      ║
# ║  - Minimal Params → fastest inference, ideal for deployment       ║
# ╚══════════════════════════════════════════════════════════════════╝

mobilenet_v3 = timm.create_model(
    CNN_MODELS['MobileNetV3-Large'],
    pretrained=USE_PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE,
)

total_p = sum(p.numel() for p in mobilenet_v3.parameters())
print(f"MobileNetV3-Large  | Params: {total_p/1e6:.1f}M | Classifier: {mobilenet_v3.classifier}")

mobilenet_v3, hist_mob, best_mob = train_model(mobilenet_v3, 'MobileNetV3', lr=CNN_LR)
plot_history(hist_mob, 'MobileNetV3-Large')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CNN Model 2: ResNeXt-50-32×4d                                   ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  Architecture:                                                    ║
# ║  - Based on ResNet-50 with grouped convolutions                   ║
# ║  - 32 groups × 4 channels each (32×4d)                           ║
# ║  - Cardinality = number of groups (new dimension beyond           ║
# ║    width/depth for scaling)                                       ║
# ║  - Each group learns independent transformations → richer         ║
# ║    feature diversity at same parameter cost                       ║
# ╚══════════════════════════════════════════════════════════════════╝

resnext50 = timm.create_model(
    CNN_MODELS['ResNeXt-50-32x4d'],
    pretrained=USE_PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE,
)

total_p = sum(p.numel() for p in resnext50.parameters())
print(f"ResNeXt-50-32x4d   | Params: {total_p/1e6:.1f}M")

resnext50, hist_rnx, best_rnx = train_model(resnext50, 'ResNeXt50', lr=CNN_LR)
plot_history(hist_rnx, 'ResNeXt-50-32x4d')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CNN Model 3: EfficientNet-B3                                     ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  Architecture:                                                    ║
# ║  - Starts from EfficientNet-B0 (NAS-found baseline)              ║
# ║  - B3 applies compound scaling: depth×1.4, width×1.2, res×300   ║
# ║  - Compound coefficient φ ensures depth/width/res scale          ║
# ║    proportionally (not independently)                             ║
# ║  - Mobile Inverted Bottleneck Convolution (MBConv) blocks        ║
# ║  - Note: EfficientNetV2/V3 are improvements; B3 has well-tested  ║
# ║    pretrained weights available in timm offline                   ║
# ╚══════════════════════════════════════════════════════════════════╝

efficientnet_b3 = timm.create_model(
    CNN_MODELS['EfficientNet-B3'],
    pretrained=USE_PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE,
)

total_p = sum(p.numel() for p in efficientnet_b3.parameters())
print(f"EfficientNet-B3    | Params: {total_p/1e6:.1f}M")

efficientnet_b3, hist_efn, best_efn = train_model(efficientnet_b3, 'EfficientNetB3', lr=CNN_LR)
plot_history(hist_efn, 'EfficientNet-B3')

## 📈 Evaluation & Comparison <a id='evaluation'></a>

In [ ]:
# ── Evaluate CNN models ────────────────────────────────────────────
CNN_MODELS = {
    'EfficientNet-B3': 'efficientnet_b3',
}

ALL_MODELS = {
    'MobileNetV3-Large': mobilenet_v3,
    'ResNeXt-50-32x4d':  resnext50,
    'EfficientNet-B3':   efficientnet_b3,
}
ALL_HISTORIES = {
    'MobileNetV3-Large': hist_mob,
    'ResNeXt-50-32x4d':  hist_rnx,
    'EfficientNet-B3':   hist_efn,
}

results = {}
crit_eval = nn.CrossEntropyLoss()

print(f"{'Model':<22} | {'Acc':>6} | {'F1':>6} | {'AUC':>6}")
print('-'*48)
for name, model in ALL_MODELS.items():
    _, acc, preds, labels, probs = evaluate(model, test_loader, crit_eval)
    lbin = label_binarize(labels, classes=range(NUM_CLASSES))
    auc_s = roc_auc_score(lbin, probs, multi_class='ovr', average='macro') * 100
    f1_s  = f1_score(labels, preds, average='macro') * 100
    results[name] = dict(accuracy=acc, f1=f1_s, auc=auc_s,
                          preds=preds, labels=labels, probs=probs)
    print(f"{name:<22} | {acc:>5.2f}% | {f1_s:>5.2f}% | {auc_s:>5.2f}%")

BEST_NAME = max(results, key=lambda k: results[k]['accuracy'])
print(f"\n🏆 Best CNN: {BEST_NAME}  ({results[BEST_NAME]['accuracy']:.2f}%)")


In [ ]:
# ── Comparative bar charts ─────────────────────────────────────────
names = list(results.keys())
accs  = [results[m]['accuracy'] for m in names]
f1s   = [results[m]['f1']       for m in names]
aucs  = [results[m]['auc']      for m in names]
CNN_NAMES = list(CNN_MODELS.keys())
bcolors = ['#3498DB' if n in CNN_NAMES else '#E74C3C' for n in names]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
for ax, vals, title in zip(axes, [accs, f1s, aucs],
                            ['Test Accuracy (%)', 'Macro F1 (%)', 'Macro AUC (%)']):
    bars = ax.bar(names, vals, color=bcolors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(max(0, min(vals)-10), 100)
    ax.tick_params(axis='x', rotation=35)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+.3,
                f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')
    ax.legend(handles=[
        mpatches.Patch(color='#3498DB', label='CNN'),
        mpatches.Patch(color='#E74C3C', label='Transformer')
    ], fontsize=8)
    ax.grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')

for ax, name in zip(axes.flat, names):
    cm  = confusion_matrix(results[name]['labels'], results[name]['preds'])
    cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=cm, fmt='d', ax=ax, cmap='Blues', vmin=0, vmax=1,
                xticklabels=short, yticklabels=short, cbar=False, linewidths=.5)
    ax.set_title(f'{name}  ({results[name]["accuracy"]:.1f}%)', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrices.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Classification report + per-class bar ─────────────────────────
br = results[BEST_NAME]
print(f"\n{'='*60}\nClassification Report — {BEST_NAME}\n{'='*60}")
print(classification_report(br['labels'], br['preds'], target_names=CLASS_NAMES, digits=4))

rep = classification_report(br['labels'], br['preds'],
                              target_names=CLASS_NAMES, output_dict=True)
fig, ax = plt.subplots(figsize=(12, 5))
x, w = np.arange(NUM_CLASSES), 0.25
ax.bar(x-w,   [rep[c]['precision'] for c in CLASS_NAMES], w, label='Precision', color='#3498DB', alpha=.8)
ax.bar(x,     [rep[c]['recall']    for c in CLASS_NAMES], w, label='Recall',    color='#E74C3C', alpha=.8)
ax.bar(x+w,   [rep[c]['f1-score']  for c in CLASS_NAMES], w, label='F1',        color='#2ECC71', alpha=.8)
ax.set_xticks(x); ax.set_xticklabels(short, rotation=15)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=.3)
ax.set_title(f'Per-Class Metrics — {BEST_NAME}', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/per_class_metrics.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('ROC Curves — One-vs-Rest (OvR)', fontsize=14, fontweight='bold')

for ax, name in zip(axes.flat, names):
    lbin = label_binarize(results[name]['labels'], classes=range(NUM_CLASSES))
    probs = results[name]['probs']
    for ci, cn in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(lbin[:,ci], probs[:,ci])
        ax.plot(fpr, tpr, color=COLORS[ci], lw=2,
                label=f'{short[ci]} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--',alpha=.4); ax.grid(alpha=.3)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{name}  (AUC={results[name]["auc"]:.1f}%)', fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_curves.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── Convergence comparison ─────────────────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Training Convergence Comparison', fontsize=13, fontweight='bold')
styles = ['-','--',':','-.', '-','--']
pcolors = ['#3498DB','#1ABC9C','#2980B9','#E74C3C','#C0392B','#E67E22']

for (name, h), ls, pc in zip(ALL_HISTORIES.items(), styles, pcolors):
    ep = range(1, len(h['val_acc'])+1)
    a1.plot(ep, h['train_acc'], ls=ls, color=pc, alpha=.4)
    a1.plot(ep, h['val_acc'],   ls=ls, color=pc, lw=2, label=name)
    a2.plot(ep, h['val_loss'],  ls=ls, color=pc, lw=2, label=name)

a1.set_title('Val Accuracy (solid) / Train Accuracy (faded)')
a1.set_xlabel('Epoch'); a1.set_ylabel('Accuracy (%)')
a1.legend(fontsize=8); a1.grid(alpha=.3)
a2.set_title('Validation Loss'); a2.set_xlabel('Epoch')
a2.set_ylabel('Loss'); a2.legend(fontsize=8); a2.grid(alpha=.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/convergence.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

## 🛡️ Robustness Analysis <a id='robustness'></a>

> **Motivation**: A model deployed in the field (e.g., on a farmer's phone) encounters dirty lenses,
> bad lighting, compression artefacts. We simulate these corruptions systematically.
>
> Corruption severity levels are controlled by `NOISE_LEVELS` and `BLUR_KERNELS` in Config.

In [ ]:
# ── Build corruption transforms from Config ────────────────────────
def make_noise_transform(sigma):
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.Lambda(lambda x: x + torch.randn_like(x) * sigma)
    ])

def make_blur_transform(k):
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.GaussianBlur(kernel_size=k if k % 2 == 1 else k+1, sigma=k/2),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

CORRUPTIONS = {'✅ Clean (Baseline)': val_transform}
for s in NOISE_LEVELS:
    CORRUPTIONS[f'Gaussian Noise σ={s}'] = make_noise_transform(s)
for k in BLUR_KERNELS:
    CORRUPTIONS[f'Gaussian Blur k={k}'] = make_blur_transform(k)
CORRUPTIONS['Overexposure (bright)'] = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ColorJitter(brightness=(1.8,1.8)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
CORRUPTIONS['Low Contrast'] = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ColorJitter(contrast=(0.3,0.3)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
CORRUPTIONS['Rotation 45°'] = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomRotation(degrees=(45,45)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

best_model = ALL_MODELS[BEST_NAME]
rob_results = {}

print(f"⏳ Robustness testing on: {BEST_NAME}")
for cname, ctransform in CORRUPTIONS.items():
    cds = datasets.ImageFolder(str(TEST_PATH), transform=ctransform)
    cld = DataLoader(cds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    _, acc, preds, lbls, _ = evaluate(best_model, cld)
    f1_ = f1_score(lbls, preds, average='macro') * 100
    rob_results[cname] = {'accuracy': acc, 'f1': f1_}
    print(f"  {cname:<30} Acc={acc:.2f}%  F1={f1_:.2f}%")

In [ ]:
# ── Robustness visualization ───────────────────────────────────────
base_acc = rob_results['✅ Clean (Baseline)']['accuracy']
rnames = list(rob_results.keys())
raccs  = [rob_results[n]['accuracy'] for n in rnames]
drops  = [base_acc - a for a in raccs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Robustness Analysis — {BEST_NAME}', fontsize=13, fontweight='bold')

rcolors = ['#27AE60' if n == '✅ Clean (Baseline)'
           else ('#E74C3C' if a < base_acc-15 else '#F39C12')
           for n, a in zip(rnames, raccs)]
bars = ax1.bar(range(len(rnames)), raccs, color=rcolors)
ax1.axhline(base_acc, color='green', ls='--', lw=2, alpha=.7, label=f'Baseline {base_acc:.1f}%')
ax1.set_xticks(range(len(rnames))); ax1.set_xticklabels(rnames, rotation=40, ha='right', fontsize=8)
ax1.set_ylabel('Accuracy (%)'); ax1.set_title('Accuracy Under Corruptions')
ax1.legend(); ax1.set_ylim(0, 105); ax1.grid(axis='y', alpha=.3)
for b, a in zip(bars, raccs):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+.5,
             f'{a:.1f}', ha='center', fontsize=8, fontweight='bold')

dcols = ['#27AE60' if d<5 else ('#F39C12' if d<15 else '#E74C3C') for d in drops]
ax2.barh(rnames, drops, color=dcols)
ax2.axvline(0, color='k', lw=.8)
ax2.axvline(5,  color='orange', ls='--', alpha=.5, label='5pp')
ax2.axvline(15, color='red',    ls='--', alpha=.5, label='15pp')
ax2.set_xlabel('Accuracy Drop (pp)'); ax2.set_title('Degradation vs Baseline')
ax2.legend(fontsize=8); ax2.grid(axis='x', alpha=.3)
for i, d in enumerate(drops):
    ax2.text(d+.2, i, f'{d:.1f}pp', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/robustness.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── GradCAM++ Explainability ───────────────────────────────────────
"""
GradCAM++ uses second-order gradients for better localization.
  heatmap = ReLU( Σ_k α_k^c · A_k )   where α are importance weights
Overlaying on the image shows WHICH pixels drive the prediction.
"""

def get_target_layer(model, name):
    if 'MobileNet' in name:
        return [model.conv_head]
    elif 'ResNeXt' in name:
        return [model.layer4[-1].conv3]
    elif 'EfficientNet' in name:
        return [model.conv_head]
    elif 'ViT' in name or 'DeiT' in name:
        return [model.blocks[-1].norm1]
    elif 'Swin' in name:
        return [model.layers[-1].blocks[-1].norm1]
    return [list(model.children())[-2]]

def raw_and_tensor(img_path):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    raw = np.array(img) / 255.0
    t   = val_transform(img).unsqueeze(0).to(DEVICE)
    return raw, t

try:
    cam = GradCAMPlusPlus(model=best_model, target_layers=get_target_layer(best_model, BEST_NAME))
    n   = GRADCAM_SAMPLES_PER_CLASS
    fig, axes = plt.subplots(NUM_CLASSES*n, 3, figsize=(10, NUM_CLASSES*n*3))
    fig.suptitle(f'GradCAM++ — {BEST_NAME}  (Original | Map | Overlay)', fontsize=12, fontweight='bold')

    row = 0
    for ci, cls in enumerate(CLASS_NAMES):
        for p in sample_imgs(TEST_PATH, cls, n):
            raw, t = raw_and_tensor(p)
            with torch.no_grad():
                out  = best_model(t)
                pred = out.argmax(1).item()
                conf = F.softmax(out,1)[0,pred].item()
            gcam = cam(t, [ClassifierOutputTarget(pred)])[0]
            over = show_cam_on_image(raw.astype(np.float32), gcam, use_rgb=True)

            tick = '✅' if pred == ci else '❌'
            axes[row,0].imshow(raw);     axes[row,0].set_title(f'True: {short[ci]}', fontsize=8)
            axes[row,1].imshow(gcam, cmap='jet'); axes[row,1].set_title('GradCAM++ Map', fontsize=8)
            axes[row,2].imshow(over);
            axes[row,2].set_title(f'{tick} {short[pred]} ({conf:.0%})', fontsize=8)
            for ax in axes[row]: ax.axis('off')
            row += 1

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/gradcam.png', dpi=FIGDPI, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"GradCAM skipped: {e}")

## 🔍 Error Analysis <a id='error'></a>

In [ ]:
# ── Confidence & calibration analysis ─────────────────────────────
preds_  = results[BEST_NAME]['preds']
labels_ = results[BEST_NAME]['labels']
probs_  = results[BEST_NAME]['probs']
maxp    = probs_.max(axis=1)
correct = (preds_ == labels_)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Error Analysis — {BEST_NAME}', fontsize=13, fontweight='bold')

# Confidence distributions
axes[0,0].hist(maxp[correct],  bins=25, alpha=.7, color='green', label='Correct', density=True)
axes[0,0].hist(maxp[~correct], bins=25, alpha=.7, color='red',   label='Wrong',   density=True)
axes[0,0].set_xlabel('Confidence'); axes[0,0].set_ylabel('Density')
axes[0,0].set_title('Confidence: Correct vs Wrong')
axes[0,0].legend(); axes[0,0].axvline(.5, color='gray', ls='--', alpha=.5)

# Calibration curve
bins = np.linspace(0,1,11)
bc = (bins[:-1]+bins[1:])/2
cal_acc = []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (maxp>=lo)&(maxp<hi)
    cal_acc.append(correct[m].mean() if m.sum()>0 else 0)
axes[0,1].plot([0,1],[0,1],'k--',alpha=.5,label='Perfect')
axes[0,1].bar(bc, cal_acc, width=.09, alpha=.7, color='steelblue', label='Model')
axes[0,1].set_title('Calibration (Reliability Diagram)')
axes[0,1].set_xlabel('Confidence'); axes[0,1].set_ylabel('Accuracy')
axes[0,1].legend(); axes[0,1].set_xlim(0,1); axes[0,1].set_ylim(0,1)

# Per-class error rate
cls_errs = [1-correct[labels_==i].mean() if (labels_==i).sum()>0 else 0
            for i in range(NUM_CLASSES)]
bars = axes[1,0].bar(short, [e*100 for e in cls_errs], color=COLORS)
axes[1,0].set_ylabel('Error Rate (%)'); axes[1,0].set_title('Per-Class Error Rate')
axes[1,0].tick_params(axis='x', rotation=20); axes[1,0].grid(axis='y', alpha=.3)
for b, e in zip(bars, cls_errs):
    axes[1,0].text(b.get_x()+b.get_width()/2, b.get_height()+.3,
                   f'{e*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Top confusion pairs
cm = confusion_matrix(labels_, preds_)
pairs = [(CLASS_NAMES[i].split()[0]+' → '+CLASS_NAMES[j].split()[0], cm[i,j])
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i!=j and cm[i,j]>0]
pairs.sort(key=lambda x: -x[1])
if pairs:
    pair_names, pair_counts = zip(*pairs[:8])
    axes[1,1].barh(pair_names, pair_counts, color='#E74C3C', alpha=.8)
    axes[1,1].set_xlabel('Count'); axes[1,1].set_title('Top Confusion Pairs')
    axes[1,1].grid(axis='x', alpha=.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/error_analysis.png', dpi=FIGDPI, bbox_inches='tight')
plt.show()

In [ ]:
# ── High-confidence wrong predictions ─────────────────────────────
test_no_aug = datasets.ImageFolder(str(TEST_PATH), transform=val_transform)
test_ldr1   = DataLoader(test_no_aug, batch_size=1, shuffle=False)

hard_errors = []
with torch.no_grad():
    for i, (img, lbl) in enumerate(test_ldr1):
        out  = best_model(img.to(DEVICE))
        prob = F.softmax(out,1)[0]
        pred = out.argmax(1).item()
        conf = prob[pred].item()
        if pred != lbl.item() and conf > 0.7:
            hard_errors.append({'conf':conf,'true':lbl.item(),'pred':pred,'img':img[0].clone()})

hard_errors.sort(key=lambda x: -x['conf'])
print(f"High-confidence errors (>70% wrong): {len(hard_errors)}")

n_show = min(12, len(hard_errors))
if n_show > 0:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle(f'High-Confidence Errors — {BEST_NAME}', fontsize=12, fontweight='bold')
    for idx, err in enumerate(hard_errors[:n_show]):
        r, c = idx//4, idx%4
        axes[r,c].imshow(denorm(err['img']).permute(1,2,0).numpy())
        axes[r,c].set_title(
            f"True: {short[err['true']]}\nPred: {short[err['pred']]} ({err['conf']:.0%})",
            fontsize=8, color='red')
        axes[r,c].axis('off')
    for idx in range(n_show, 12):
        axes[idx//4, idx%4].axis('off')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/high_conf_errors.png', dpi=FIGDPI, bbox_inches='tight')
    plt.show()
else:
    print("✅ No high-confidence errors — great calibration!")

In [ ]:
# ── Final summary table ────────────────────────────────────────────
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

sum_rows = []
for name, m in ALL_MODELS.items():
    p = sum(x.numel() for x in m.parameters())
    sum_rows.append({
        'Model': name,
        'Type': 'CNN' if name in CNN_MODELS else 'Transformer',
        'Params': f'{p/1e6:.1f}M',
        'Accuracy': f"{results[name]['accuracy']:.2f}%",
        'F1 Macro': f"{results[name]['f1']:.2f}%",
        'AUC':      f"{results[name]['auc']:.2f}%",
    })
print(pd.DataFrame(sum_rows).to_string(index=False))
print(f"\n🏆 Best model: {BEST_NAME} ({results[BEST_NAME]['accuracy']:.2f}%)")

print("""
╔══════════════════════════════════════════════════════════════════╗
║  KEY TAKEAWAYS FOR STUDENTS                                       ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. Transfer learning >> Random init, especially with small data  ║
║  2. Augmentation is your free lunch — use it aggressively         ║
║  3. CNNs still competitive (especially MobileNet) for small sets  ║
║  4. Transformers shine when data > ~100k images (or pretrained)   ║
║  5. Label smoothing: prevents overconfident/miscalibrated models  ║
║  6. GradCAM is essential — always check WHERE the model looks     ║
║  7. Robustness matters more than clean-test accuracy in practice  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---

## 📚 References

1. **MobileNetV3**: Howard et al. (2019). *Searching for MobileNetV3*. ICCV 2019.
2. **ResNeXt**: Xie et al. (2017). *Aggregated Residual Transformations for Deep Neural Networks*. CVPR 2017.
3. **EfficientNet**: Tan & Le (2019). *EfficientNet: Rethinking Model Scaling for CNNs*. ICML 2019.
4. **ViT**: Dosovitskiy et al. (2020). *An Image is Worth 16x16 Words*. ICLR 2021.
5. **Swin**: Liu et al. (2021). *Swin Transformer: Hierarchical ViT using Shifted Windows*. ICCV 2021.
6. **DeiT**: Touvron et al. (2021). *Training data-efficient image transformers*. ICML 2021.
7. **GradCAM++**: Chattopadhyay et al. (2018). *Grad-CAM++*. WACV 2018.
8. **Label Smoothing**: Müller et al. (2019). *When Does Label Smoothing Help?*. NeurIPS 2019.
9. **AdamW**: Loshchilov & Hutter (2019). *Decoupled Weight Decay Regularization*. ICLR 2019.
10. **RandAugment**: Cubuk et al. (2020). *RandAugment: Practical automated data augmentation*. CVPR 2020.

---
*East West University — CSE 445 / CSE 475*